In [ ]:
import pandas as pd
import feedparser
import requests
import re
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import time

In [6]:
urls = {
    "Avis": "https://www.cert.ssi.gouv.fr/avis/feed/",
    "Alertes": "https://www.cert.ssi.gouv.fr/alerte/feed/"
}

bulletins = []

for type_bulletin, url in urls.items():
    rss_feed = feedparser.parse(url)
    for entry in rss_feed.entries:
        bulletins.append({
            "id_bulletin": entry.link.split('/')[-2], # Extrait l'ID (ex: CERTFR-2024-ALE-001)
            "titre": entry.title,
            "type": type_bulletin,
            "date_publication": entry.published,
            "lien": entry.link
        })

print(f"{len(bulletins)} bulletins extraits avec succès.")
print("Exemple de bulletin :", bulletins[0])

80 bulletins extraits avec succès.
Exemple de bulletin : {'id_bulletin': 'CERTFR-2026-AVI-0692', 'titre': 'Multiples vulnérabilités dans Google Chrome (05 juin 2026)', 'type': 'Avis', 'date_publication': 'Fri, 05 Jun 2026 00:00:00 +0000', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0692/'}


In [4]:

url = "https://www.cert.ssi.gouv.fr/alerte/CERTFR-2024-ALE-001/json/"
response = requests.get(url)
data = response.json()
#Extraction des CVE reference dans la clé cves du dict data
ref_cves=list(data["cves"])
#attention il s’agit d’une liste des dictionnaires avec name et url comme clés
print( "CVE référencés ", ref_cves)
# Extraction des CVE avec une regex
cve_pattern = r"CVE-\d{4}-\d{4,7}"
cve_list = list(set(re.findall(cve_pattern, str(data))))
print("CVE trouvés :", cve_list)


CVE référencés  [{'name': 'CVE-2023-46805', 'url': 'https://www.cve.org/CVERecord?id=CVE-2023-46805'}, {'name': 'CVE-2024-21887', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-21887'}, {'name': 'CVE-2024-21893', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-21893'}, {'name': 'CVE-2024-21888', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-21888'}, {'name': 'CVE-2024-22024', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-22024'}]
CVE trouvés : ['CVE-2024-21893', 'CVE-2024-21888', 'CVE-2024-21887', 'CVE-2023-46805', 'CVE-2024-22024']


In [8]:
bulletins_list = []
for b in bulletins:
    json_url = b["lien"] 
    json_url += "json/"
    response = requests.get(json_url)
    data = response.json()
    #Extraction des CVE reference dans la clé cves du dict data
    ref_cves=list(data["cves"])
    #attention il s’agit d’une liste des dictionnaires avec name et url comme clés
    # Extraction des CVE avec une regex
    cve_pattern = r"CVE-\d{4}-\d{4,7}"
    cve_list = list(set(re.findall(cve_pattern, str(data))))
    print("CVE trouvés :", cve_list)
    for cve in cve_list:
                bulletin_cve = b.copy()
                bulletin_cve["cve_id"] = cve
                bulletins_list.append(bulletin_cve)
    
print(f"{len(bulletins_list)} bulletins enrichis avec CVE extraits.")
print("Exemple de bulletin enrichi :", bulletins_list[0])

CVE trouvés : ['CVE-2026-11077', 'CVE-2026-11170', 'CVE-2026-10925', 'CVE-2026-11136', 'CVE-2026-11086', 'CVE-2026-11035', 'CVE-2026-11044', 'CVE-2026-11065', 'CVE-2026-11100', 'CVE-2026-11271', 'CVE-2026-10893', 'CVE-2026-11210', 'CVE-2026-11196', 'CVE-2026-11131', 'CVE-2026-11004', 'CVE-2026-10997', 'CVE-2026-11032', 'CVE-2026-11245', 'CVE-2026-11227', 'CVE-2026-10964', 'CVE-2026-11215', 'CVE-2026-11117', 'CVE-2026-11269', 'CVE-2026-11201', 'CVE-2026-10908', 'CVE-2026-11067', 'CVE-2026-11087', 'CVE-2026-11256', 'CVE-2026-11286', 'CVE-2026-11176', 'CVE-2026-11243', 'CVE-2026-11266', 'CVE-2026-11007', 'CVE-2026-10902', 'CVE-2026-11031', 'CVE-2026-11212', 'CVE-2026-11290', 'CVE-2026-11172', 'CVE-2026-10990', 'CVE-2026-11088', 'CVE-2026-11192', 'CVE-2026-11059', 'CVE-2026-10915', 'CVE-2026-10963', 'CVE-2026-11189', 'CVE-2026-10979', 'CVE-2026-11155', 'CVE-2026-11199', 'CVE-2026-11142', 'CVE-2026-11039', 'CVE-2026-11181', 'CVE-2026-10965', 'CVE-2026-10927', 'CVE-2026-10896', 'CVE-2026-109

In [9]:
print("Exemple de bulletin enrichi :", bulletins_list[0:4])

Exemple de bulletin enrichi : [{'id_bulletin': 'CERTFR-2026-AVI-0692', 'titre': 'Multiples vulnérabilités dans Google Chrome (05 juin 2026)', 'type': 'Avis', 'date_publication': 'Fri, 05 Jun 2026 00:00:00 +0000', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0692/', 'cve_id': 'CVE-2026-11077'}, {'id_bulletin': 'CERTFR-2026-AVI-0692', 'titre': 'Multiples vulnérabilités dans Google Chrome (05 juin 2026)', 'type': 'Avis', 'date_publication': 'Fri, 05 Jun 2026 00:00:00 +0000', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0692/', 'cve_id': 'CVE-2026-11170'}, {'id_bulletin': 'CERTFR-2026-AVI-0692', 'titre': 'Multiples vulnérabilités dans Google Chrome (05 juin 2026)', 'type': 'Avis', 'date_publication': 'Fri, 05 Jun 2026 00:00:00 +0000', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0692/', 'cve_id': 'CVE-2026-10925'}, {'id_bulletin': 'CERTFR-2026-AVI-0692', 'titre': 'Multiples vulnérabilités dans Google Chrome (05 juin 2026)', 'type': 'Avis', 'date_pu